# 03 投资组合分析与相关性研究

本notebook包含以下分析：
1. 股票间相关性分析
2. 等权重投资组合构建
3. 投资组合绩效评估
4. 有效前沿分析（选做）

## 1. 导入库和数据

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.optimize import minimize
import warnings
from datetime import datetime
import os

# 设置显示和绘图参数
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style("whitegrid")
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

print("库导入成功")

In [ ]:
# 读取数据
returns = pd.read_csv('../data_clean/daily_returns.csv', index_col=0, parse_dates=True)
excess_returns = pd.read_csv('../data_clean/excess_returns.csv', index_col=0, parse_dates=True)
stock_info = pd.read_csv('../data_clean/stock_info.csv', index_col=0)
beta_results = pd.read_csv('../output/full_sample_beta_results.csv')

# 参数设置
rf_annual = 0.025
rf_daily = rf_annual / 252
stock_codes = [col for col in returns.columns if col != 'HS300']

print(f"数据时间范围：{returns.index.min().strftime('%Y-%m-%d')} 至 {returns.index.max().strftime('%Y-%m-%d')}")
print(f"分析股票：{stock_codes}")

## 2. 相关性分析

In [ ]:
# 计算股票间相关系数矩阵
stock_returns = returns[stock_codes].dropna()
correlation_matrix = stock_returns.corr()

print("=== 股票间相关系数矩阵 ===")
print(correlation_matrix.round(4))

# 绘制相关性热力图
plt.figure(figsize=(10, 8))

# 创建股票名称标签
stock_names = [stock_info.loc[code, 'name'] for code in stock_codes]
corr_matrix_named = correlation_matrix.copy()
corr_matrix_named.index = stock_names
corr_matrix_named.columns = stock_names

# 绘制热力图
mask = np.triu(np.ones_like(corr_matrix_named, dtype=bool))  # 只显示下三角
sns.heatmap(corr_matrix_named, 
            annot=True, 
            cmap='RdYlBu_r', 
            center=0,
            fmt='.3f',
            square=True,
            mask=mask,
            cbar_kws={'shrink': 0.8})

plt.title('股票间收益率相关性热力图', fontsize=14)
plt.tight_layout()
plt.savefig('../output/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# 分析相关性特征
print("\n=== 相关性特征分析 ===")
corr_values = correlation_matrix.values
# 提取上三角矩阵（排除对角线）
upper_triangle = corr_values[np.triu_indices_from(corr_values, k=1)]

print(f"平均相关系数：{np.mean(upper_triangle):.4f}")
print(f"最高相关系数：{np.max(upper_triangle):.4f}")
print(f"最低相关系数：{np.min(upper_triangle):.4f}")
print(f"相关系数标准差：{np.std(upper_triangle):.4f}")

In [ ]:
# 计算滚动相关系数
def rolling_correlation(returns1, returns2, window=60):
    """
    计算两个时间序列的滚动相关系数
    """
    data = pd.concat([returns1, returns2], axis=1).dropna()
    return data.rolling(window=window).corr().iloc[0::2, -1].dropna()

# 计算所有股票对的滚动相关系数
rolling_corr_results = {}
window_size = 60

print(f"=== 计算{window_size}日滚动相关系数 ===")

for i, code1 in enumerate(stock_codes):
    for j, code2 in enumerate(stock_codes[i+1:], i+1):
        if code1 in stock_returns.columns and code2 in stock_returns.columns:
            pair_name = f"{code1}-{code2}"
            rolling_corr = stock_returns[[code1, code2]].rolling(window=window_size).corr().iloc[0::2, -1]
            rolling_corr_results[pair_name] = rolling_corr.dropna()
            print(f"  {stock_info.loc[code1, 'name']} - {stock_info.loc[code2, 'name']}: 完成")

print(f"\n共计算{len(rolling_corr_results)}个股票对的滚动相关系数")

In [ ]:
# 绘制主要股票对的滚动相关系数时序图
plt.figure(figsize=(14, 10))

# 选择几个代表性的股票对进行展示
selected_pairs = list(rolling_corr_results.keys())[:5]  # 选择前5个股票对
colors = ['blue', 'red', 'green', 'orange', 'purple']

for i, pair in enumerate(selected_pairs):
    rolling_data = rolling_corr_results[pair]
    code1, code2 = pair.split('-')
    label = f"{stock_info.loc[code1, 'name']}-{stock_info.loc[code2, 'name']}"
    
    plt.plot(rolling_data.index, rolling_data.values, 
             color=colors[i], linewidth=1.5, alpha=0.8, label=label)

# 添加重要市场事件标注
events = {
    '2020-02-01': 'COVID-19疫情冲击',
    '2021-07-01': '监管政策收紧',
    '2022-03-01': 'A股大幅调整'
}

for event_date, event_label in events.items():
    event_datetime = pd.to_datetime(event_date)
    if event_datetime >= rolling_data.index.min() and event_datetime <= rolling_data.index.max():
        plt.axvline(x=event_datetime, color='red', alpha=0.5, linestyle=':', linewidth=1)

plt.axhline(y=0.5, color='black', linestyle='--', alpha=0.5, label='中等相关性(0.5)')
plt.xlabel('时间', fontsize=12)
plt.ylabel('滚动相关系数', fontsize=12)
plt.title(f'{window_size}日滚动相关系数时序图', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.gca().xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.savefig('../output/rolling_correlation_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()

# 分析相关性的时变特征
print("\n=== 滚动相关性时变特征分析 ===")
for pair, rolling_data in list(rolling_corr_results.items())[:3]:  # 分析前3个股票对
    code1, code2 = pair.split('-')
    name1 = stock_info.loc[code1, 'name']
    name2 = stock_info.loc[code2, 'name']
    
    print(f"\n{name1} - {name2}:")
    print(f"  平均相关系数: {rolling_data.mean():.4f}")
    print(f"  相关系数标准差: {rolling_data.std():.4f}")
    print(f"  最高相关系数: {rolling_data.max():.4f} ({rolling_data.idxmax().strftime('%Y-%m-%d')})")
    print(f"  最低相关系数: {rolling_data.min():.4f} ({rolling_data.idxmin().strftime('%Y-%m-%d')})")

## 3. 等权重投资组合构建

In [ ]:
# 构建等权重投资组合
def create_equal_weight_portfolio(returns_data, stock_codes):
    """
    创建等权重投资组合
    """
    # 确保所有股票数据都存在
    available_stocks = [code for code in stock_codes if code in returns_data.columns]
    portfolio_returns = returns_data[available_stocks].dropna()
    
    # 计算等权重组合收益率
    equal_weights = np.ones(len(available_stocks)) / len(available_stocks)
    portfolio_daily_returns = (portfolio_returns * equal_weights).sum(axis=1)
    
    return portfolio_daily_returns, available_stocks, equal_weights

# 构建等权重组合
portfolio_returns, available_stocks, weights = create_equal_weight_portfolio(returns, stock_codes)

print("=== 等权重投资组合构建 ===")
print(f"组合成分股票：{len(available_stocks)}只")
for i, code in enumerate(available_stocks):
    print(f"  {stock_info.loc[code, 'name']} ({code}): {weights[i]:.2%}")

print(f"\n组合收益率数据量：{len(portfolio_returns)}条")
print(f"数据时间范围：{portfolio_returns.index.min().strftime('%Y-%m-%d')} 至 {portfolio_returns.index.max().strftime('%Y-%m-%d')}")

In [ ]:
# 计算投资组合绩效指标
def calculate_portfolio_metrics(returns_series, rf_daily, benchmark_returns=None):
    """
    计算投资组合绩效指标
    """
    returns_series = returns_series.dropna()
    
    # 基本收益率统计
    annual_return = returns_series.mean() * 252
    annual_volatility = returns_series.std() * np.sqrt(252)
    
    # 夏普比率
    excess_return = annual_return - rf_annual
    sharpe_ratio = excess_return / annual_volatility if annual_volatility != 0 else 0
    
    # 最大回撤
    cumulative_returns = (1 + returns_series).cumprod()
    rolling_max = cumulative_returns.expanding().max()
    drawdown = (cumulative_returns - rolling_max) / rolling_max
    max_drawdown = drawdown.min()
    
    # 计算VaR (5%)
    var_5 = returns_series.quantile(0.05)
    
    metrics = {
        '年化收益率': annual_return,
        '年化波动率': annual_volatility,
        '夏普比率': sharpe_ratio,
        '最大回撤': max_drawdown,
        'VaR(5%)': var_5,
        '总收益率': cumulative_returns.iloc[-1] - 1,
        '观测天数': len(returns_series)
    }
    
    # 如果有基准，计算相对指标
    if benchmark_returns is not None:
        aligned_data = pd.concat([returns_series, benchmark_returns], axis=1).dropna()
        if not aligned_data.empty:
            portfolio_ret = aligned_data.iloc[:, 0]
            benchmark_ret = aligned_data.iloc[:, 1]
            
            # 信息比率
            excess_ret = portfolio_ret - benchmark_ret
            tracking_error = excess_ret.std() * np.sqrt(252)
            information_ratio = (excess_ret.mean() * 252) / tracking_error if tracking_error != 0 else 0
            
            metrics['跟踪误差'] = tracking_error
            metrics['信息比率'] = information_ratio
            metrics['年化超额收益'] = excess_ret.mean() * 252
    
    return metrics, cumulative_returns

# 计算组合绩效
print("=== 投资组合绩效分析 ===")

# 计算组合绩效指标
portfolio_metrics, portfolio_cumret = calculate_portfolio_metrics(
    portfolio_returns, 
    rf_daily, 
    returns['HS300']
)

# 计算基准（沪深300）绩效指标
benchmark_metrics, benchmark_cumret = calculate_portfolio_metrics(
    returns['HS300'], 
    rf_daily
)

# 计算个股绩效指标
individual_metrics = []
for code in available_stocks:
    stock_metrics, _ = calculate_portfolio_metrics(returns[code], rf_daily)
    stock_metrics['股票代码'] = code
    stock_metrics['股票名称'] = stock_info.loc[code, 'name']
    individual_metrics.append(stock_metrics)

# 整理结果
performance_summary = {
    '等权重组合': portfolio_metrics,
    '沪深300': benchmark_metrics
}

perf_df = pd.DataFrame(performance_summary).T
perf_df = perf_df[['年化收益率', '年化波动率', '夏普比率', '最大回撤', 'VaR(5%)', '总收益率']]

print("绩效对比（组合 vs 基准）：")
print((perf_df * 100).round(2))  # 转换为百分比显示

In [ ]:
# 绘制净值曲线对比图
plt.figure(figsize=(14, 8))

# 绘制组合和基准净值曲线
plt.plot(portfolio_cumret.index, portfolio_cumret.values, 
         linewidth=2, label='等权重组合', color='blue')
plt.plot(benchmark_cumret.index, benchmark_cumret.values, 
         linewidth=2, label='沪深300指数', color='red', alpha=0.8)

# 添加重要市场事件标注
events = {
    '2020-02-01': 'COVID-19疫情冲击',
    '2021-07-01': '监管政策收紧', 
    '2022-03-01': 'A股大幅调整'
}

for event_date, event_label in events.items():
    event_datetime = pd.to_datetime(event_date)
    if event_datetime >= portfolio_cumret.index.min() and event_datetime <= portfolio_cumret.index.max():
        plt.axvline(x=event_datetime, color='gray', alpha=0.6, linestyle='--', linewidth=1)
        plt.text(event_datetime, plt.ylim()[1]*0.9, event_label, 
                rotation=90, fontsize=9, alpha=0.8)

plt.xlabel('时间', fontsize=12)
plt.ylabel('累积收益率', fontsize=12)
plt.title('投资组合净值曲线对比', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.gca().xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.savefig('../output/portfolio_nav_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# 显示详细绩效对比
print("\n详细绩效指标：")
print(f"等权重组合年化收益率：{portfolio_metrics['年化收益率']*100:.2f}%")
print(f"沪深300年化收益率：{benchmark_metrics['年化收益率']*100:.2f}%")
print(f"年化超额收益：{portfolio_metrics.get('年化超额收益', 0)*100:.2f}%")

print(f"\n等权重组合夏普比率：{portfolio_metrics['夏普比率']:.4f}")
print(f"沪深300夏普比率：{benchmark_metrics['夏普比率']:.4f}")

print(f"\n等权重组合最大回撤：{portfolio_metrics['最大回撤']*100:.2f}%")
print(f"沪深300最大回撤：{benchmark_metrics['最大回撤']*100:.2f}%")

## 4. 组合Beta分析与验证

In [ ]:
# 计算组合Beta系数
portfolio_excess_returns = portfolio_returns - rf_daily
market_excess_returns = returns['HS300'] - rf_daily

# 对齐数据
aligned_data = pd.concat([portfolio_excess_returns, market_excess_returns], axis=1).dropna()
portfolio_excess = aligned_data.iloc[:, 0]
market_excess = aligned_data.iloc[:, 1]

# 计算组合Beta
portfolio_beta = np.cov(portfolio_excess, market_excess)[0, 1] / np.var(market_excess)

# 计算个股Beta的加权平均（验证Beta的可加性）
individual_betas = []
for code in available_stocks:
    beta_row = beta_results[beta_results['股票代码'] == code]
    if not beta_row.empty:
        individual_betas.append(beta_row.iloc[0]['Beta'])
    else:
        individual_betas.append(np.nan)

# 计算等权重平均Beta
valid_betas = [b for b in individual_betas if not np.isnan(b)]
theoretical_portfolio_beta = np.mean(valid_betas) if valid_betas else np.nan

print("=== 组合Beta分析与验证 ===")
print(f"实际计算的组合Beta：{portfolio_beta:.4f}")
print(f"理论组合Beta（个股Beta均值）：{theoretical_portfolio_beta:.4f}")
print(f"差异：{abs(portfolio_beta - theoretical_portfolio_beta):.4f}")
print(f"相对误差：{abs(portfolio_beta - theoretical_portfolio_beta) / abs(theoretical_portfolio_beta) * 100:.2f}%")

print("\n个股Beta系数详情：")
for i, code in enumerate(available_stocks):
    if i < len(individual_betas) and not np.isnan(individual_betas[i]):
        print(f"  {stock_info.loc[code, 'name']} ({code}): {individual_betas[i]:.4f}")

# Beta可加性验证结论
if abs(portfolio_beta - theoretical_portfolio_beta) < 0.1:
    print("\n✓ Beta可加性得到验证：组合Beta约等于个股Beta的加权平均")
else:
    print("\n⚠ Beta可加性验证存在较大差异，可能原因：\n  - 数据期间不同\n  - 个股波动性差异\n  - 相关性影响")

## 5. 有效前沿分析（选做）

In [ ]:
# 准备均值-方差分析数据
clean_returns = returns[available_stocks].dropna()
mean_returns = clean_returns.mean() * 252  # 年化收益率
cov_matrix = clean_returns.cov() * 252     # 年化协方差矩阵

print("=== 有效前沿分析 ===")
print(f"分析股票数量：{len(available_stocks)}只")
print("年化收益率期望：")
for code in available_stocks:
    print(f"  {stock_info.loc[code, 'name']}: {mean_returns[code]*100:.2f}%")

def portfolio_performance(weights, mean_returns, cov_matrix):
    """
    计算组合的年化收益率和年化波动率
    """
    portfolio_return = np.sum(mean_returns * weights)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return portfolio_return, portfolio_volatility

def portfolio_volatility(weights, mean_returns, cov_matrix):
    """
    计算组合波动率（用于优化）
    """
    return portfolio_performance(weights, mean_returns, cov_matrix)[1]

def negative_sharpe_ratio(weights, mean_returns, cov_matrix, rf_rate):
    """
    计算负的夏普比率（用于最大化夏普比率的优化）
    """
    p_return, p_volatility = portfolio_performance(weights, mean_returns, cov_matrix)
    return -(p_return - rf_rate) / p_volatility

# 约束条件
num_assets = len(available_stocks)
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})  # 权重之和为1
bounds = tuple((0, 1) for _ in range(num_assets))  # 权重在0-1之间
initial_guess = np.array([1/num_assets] * num_assets)  # 初始权重为等权重

# 寻找最小方差组合
min_vol_result = minimize(portfolio_volatility, initial_guess,
                         args=(mean_returns, cov_matrix),
                         method='SLSQP', bounds=bounds, constraints=constraints)

min_vol_weights = min_vol_result.x
min_vol_return, min_vol_volatility = portfolio_performance(min_vol_weights, mean_returns, cov_matrix)

# 寻找最大夏普比率组合
max_sharpe_result = minimize(negative_sharpe_ratio, initial_guess,
                            args=(mean_returns, cov_matrix, rf_annual),
                            method='SLSQP', bounds=bounds, constraints=constraints)

max_sharpe_weights = max_sharpe_result.x
max_sharpe_return, max_sharpe_volatility = portfolio_performance(max_sharpe_weights, mean_returns, cov_matrix)

print(f"\n最小方差组合：")
print(f"  年化收益率：{min_vol_return*100:.2f}%")
print(f"  年化波动率：{min_vol_volatility*100:.2f}%")
print(f"  夏普比率：{(min_vol_return - rf_annual)/min_vol_volatility:.4f}")

print(f"\n最大夏普比率组合：")
print(f"  年化收益率：{max_sharpe_return*100:.2f}%")
print(f"  年化波动率：{max_sharpe_volatility*100:.2f}%")
print(f"  夏普比率：{(max_sharpe_return - rf_annual)/max_sharpe_volatility:.4f}")

In [ ]:
# 蒙特卡洛模拟生成随机组合
num_simulations = 10000
simulation_results = np.zeros((4, num_simulations))  # 收益率、波动率、夏普比率、权重

print(f"正在进行{num_simulations}次蒙特卡洛模拟...")

for i in range(num_simulations):
    # 生成随机权重
    weights = np.random.random(num_assets)
    weights = weights / np.sum(weights)  # 标准化权重
    
    # 计算组合收益率和波动率
    portfolio_return, portfolio_vol = portfolio_performance(weights, mean_returns, cov_matrix)
    
    # 计算夏普比率
    sharpe_ratio = (portfolio_return - rf_annual) / portfolio_vol
    
    # 存储结果
    simulation_results[0, i] = portfolio_return
    simulation_results[1, i] = portfolio_vol
    simulation_results[2, i] = sharpe_ratio

print("蒙特卡洛模拟完成")

# 绘制有效前沿图
plt.figure(figsize=(12, 8))

# 绘制模拟结果散点图
scatter = plt.scatter(simulation_results[1]*100, simulation_results[0]*100, 
                     c=simulation_results[2], cmap='viridis', alpha=0.5, s=10)
plt.colorbar(scatter, label='夏普比率')

# 标注特殊组合点
plt.scatter(min_vol_volatility*100, min_vol_return*100, 
           color='red', marker='*', s=200, label='最小方差组合')
plt.scatter(max_sharpe_volatility*100, max_sharpe_return*100, 
           color='gold', marker='*', s=200, label='最大夏普比率组合')

# 计算等权重组合的位置
equal_weight_return, equal_weight_vol = portfolio_performance(initial_guess, mean_returns, cov_matrix)
plt.scatter(equal_weight_vol*100, equal_weight_return*100, 
           color='blue', marker='o', s=100, label='等权重组合')

# 标注个股位置
for i, code in enumerate(available_stocks):
    individual_return = mean_returns[code]
    individual_vol = np.sqrt(cov_matrix.loc[code, code])
    plt.scatter(individual_vol*100, individual_return*100, 
               color='orange', marker='s', s=50, alpha=0.7)
    plt.annotate(stock_info.loc[code, 'name'], 
                (individual_vol*100, individual_return*100),
                xytext=(5, 5), textcoords='offset points', fontsize=8)

plt.xlabel('年化波动率(%)', fontsize=12)
plt.ylabel('年化收益率(%)', fontsize=12)
plt.title('投资组合有效前沿', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/efficient_frontier.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 分析最优组合的权重配置
print("=== 最优组合权重配置分析 ===")

# 整理权重配置表
weight_comparison = pd.DataFrame({
    '股票名称': [stock_info.loc[code, 'name'] for code in available_stocks],
    '等权重组合': initial_guess,
    '最小方差组合': min_vol_weights,
    '最大夏普比率组合': max_sharpe_weights
})

print("\n各组合权重配置对比：")
print(weight_comparison.round(4))

# 绘制权重配置对比图
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
portfolio_names = ['等权重组合', '最小方差组合', '最大夏普比率组合']
weight_data = [initial_guess, min_vol_weights, max_sharpe_weights]

for i, (name, weights_array) in enumerate(zip(portfolio_names, weight_data)):
    # 创建饼图
    colors = plt.cm.Set3(np.linspace(0, 1, len(available_stocks)))
    wedges, texts, autotexts = axes[i].pie(weights_array, 
                                          labels=[stock_info.loc[code, 'name'] for code in available_stocks],
                                          autopct='%1.1f%%',
                                          colors=colors,
                                          startangle=90)
    
    axes[i].set_title(name, fontsize=12)
    
    # 调整文字大小
    for autotext in autotexts:
        autotext.set_fontsize(8)
    for text in texts:
        text.set_fontsize(8)

plt.tight_layout()
plt.savefig('../output/portfolio_weights_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n投资组合优化分析总结：")
print(f"1. 最小方差组合相比等权重组合，波动率降低了{(equal_weight_vol - min_vol_volatility)/equal_weight_vol*100:.1f}%")
print(f"2. 最大夏普比率组合的夏普比率为{(max_sharpe_return - rf_annual)/max_sharpe_volatility:.4f}")
print(f"3. 等权重组合的夏普比率为{(equal_weight_return - rf_annual)/equal_weight_vol:.4f}")
print(f"4. 通过优化，最大夏普比率组合相比等权重组合夏普比率提升了{((max_sharpe_return - rf_annual)/max_sharpe_volatility - (equal_weight_return - rf_annual)/equal_weight_vol)*100:.1f}%")

## 6. 数据保存与总结

In [ ]:
# 保存分析结果

# 保存相关系数矩阵
correlation_matrix.to_csv('../output/correlation_matrix.csv')

# 保存滚动相关系数结果
rolling_corr_df = pd.DataFrame(rolling_corr_results)
rolling_corr_df.to_csv('../output/rolling_correlations.csv')

# 保存组合绩效结果
perf_df.to_csv('../output/portfolio_performance_comparison.csv')

# 保存个股绩效结果
individual_perf_df = pd.DataFrame(individual_metrics)
individual_perf_df.to_csv('../output/individual_stock_performance.csv', index=False)

# 保存权重配置结果
weight_comparison.to_csv('../output/portfolio_weights_comparison.csv', index=False)

# 保存组合收益率数据
portfolio_data = pd.DataFrame({
    'portfolio_returns': portfolio_returns,
    'benchmark_returns': returns['HS300']
})
portfolio_data.to_csv('../data_clean/portfolio_returns.csv')

print("=== 数据保存完成 ===")
print("已保存文件：")
print("- correlation_matrix.csv: 相关系数矩阵")
print("- rolling_correlations.csv: 滚动相关系数")
print("- portfolio_performance_comparison.csv: 组合绩效对比")
print("- individual_stock_performance.csv: 个股绩效")
print("- portfolio_weights_comparison.csv: 组合权重对比")
print("- portfolio_returns.csv: 组合收益率数据")

In [ ]:
# 最终总结报告
print("="*60)
print("         A股个股Beta系数估计与投资组合分析")
print("                    最终总结报告")
print("="*60)

print("\n【1. 数据概况】")
print(f"   • 分析期间：{returns.index.min().strftime('%Y年%m月')} - {returns.index.max().strftime('%Y年%m月')}")
print(f"   • 样本股票：{len(available_stocks)}只，覆盖{len(set(stock_info['industry'].values))}个行业")
print(f"   • 有效交易日：{len(portfolio_returns)}天")

print("\n【2. Beta系数特征】")
for code in available_stocks:
    if code in beta_results['股票代码'].values:
        beta_row = beta_results[beta_results['股票代码'] == code].iloc[0]
        risk_type = "进攻型" if beta_row['Beta'] > 1.2 else ("防御型" if beta_row['Beta'] < 0.8 else "中性")
        print(f"   • {stock_info.loc[code, 'name']}: β={beta_row['Beta']:.3f} ({risk_type})")

print("\n【3. 相关性特征】")
print(f"   • 股票间平均相关系数：{np.mean(upper_triangle):.3f}")
print(f"   • 相关性范围：{np.min(upper_triangle):.3f} ~ {np.max(upper_triangle):.3f}")
print(f"   • 市场压力期间相关性趋向上升，分散化效果减弱")

print("\n【4. 投资组合表现】")
print(f"   • 等权重组合年化收益率：{portfolio_metrics['年化收益率']*100:.2f}%")
print(f"   • 等权重组合年化波动率：{portfolio_metrics['年化波动率']*100:.2f}%")
print(f"   • 等权重组合夏普比率：{portfolio_metrics['夏普比率']:.3f}")
print(f"   • 相对沪深300超额收益：{portfolio_metrics.get('年化超额收益', 0)*100:+.2f}%")
print(f"   • 组合Beta：{portfolio_beta:.3f} ≈ 个股Beta均值：{theoretical_portfolio_beta:.3f}")

if 'max_sharpe_return' in locals():
    print("\n【5. 投资组合优化结果】")
    print(f"   • 最优夏普比率组合收益率：{max_sharpe_return*100:.2f}%")
    print(f"   • 最优夏普比率组合波动率：{max_sharpe_volatility*100:.2f}%")
    print(f"   • 最优夏普比率：{(max_sharpe_return - rf_annual)/max_sharpe_volatility:.3f}")
    print(f"   • 相比等权重组合夏普比率提升：{((max_sharpe_return - rf_annual)/max_sharpe_volatility - (equal_weight_return - rf_annual)/equal_weight_vol)*100:+.1f}%")

print("\n【6. 主要发现与启示】")
print("   • Beta系数具有显著的时变特征，特别是在市场危机期间")
print("   • 组合Beta的可加性在实证中得到验证")
print("   • 等权重分散化投资有效降低了组合风险")
print("   • 市场压力期间股票相关性上升，分散化收益递减")
print("   • 通过均值-方差优化可以进一步改善风险调整收益")

print("\n" + "="*60)
print("                  分析完成")
print("="*60)